# CVE to CWE Mapping

## Pre reqs:
You need to have ran scrape_cwe.ipynb file and fill the CVE folder in scrapers/scraped_data in order to continue

### Requirements

In [40]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu 
%pip install sentence-transformers pandas tqdm

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [42]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search
import torch
import numpy as np
import json
import pandas as pd
from tqdm import tqdm

## Loading scraped data

### Loading CWE

In [ ]:
with open("../scrapers/scraped_data/CWE_SOFTWARE.json", "r", encoding="utf-8") as f:
    CWE = json.load(f)

### Loading CVE

In [ ]:
CVE_files = ["nvd_2025_cwe699.csv"]

CVE: pd.DataFrame = pd.DataFrame()
for i, file in enumerate(CVE_files):
    if i == 0:
        CVE = pd.read_csv(f"../scrapers/scraped_data/CVE/{file}")
        continue
    
    CVE = pd.concat([CVE, pd.read_csv(f"../scrapers/scraped_data/CVE/{file}")], ignore_index=True)

## Embedding

In [ ]:
embedding_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2511.65it/s]


In [22]:
# CWE Embeddings - Slower because not batching but it's more readable

def extract_description(child):
    description = child["description"]
    if child["extended_description"] is not None:
        description += f"\n{child["extended_description"]}"
        
    return description


for id, category in tqdm(CWE.items()):
    category["summary_embedding"] = embedding_model.encode(category["summary"])
    child_descriptions = [extract_description(child) for child in category["children"].values()]
    child_descriptions_embeddings = embedding_model.encode(child_descriptions)
    
    for index, (child_id, child) in enumerate(category["children"].items()):
        child["description_embedding"] = child_descriptions_embeddings[index]

100%|██████████| 40/40 [00:07<00:00,  5.31it/s]


In [31]:
# Checking for embeding description values
if len(CVE[CVE["description"].notna() == False]) != 0:
    raise ValueError("Some descriptions in CVE are empty, please check and remove")

In [35]:
# CVE Embeddings
embeddings = embedding_model.encode(
    CVE["description"].tolist(), batch_size=64, show_progress_bar=True
)

Batches:   0%|          | 0/419 [00:00<?, ?it/s]

Batches: 100%|██████████| 419/419 [05:27<00:00,  1.28it/s]


In [37]:
CVE["description_embeddings"] = list(embeddings)

## Cosine similarity for Categories

In [60]:
category_embeddings = [category["summary_embedding"] for category in CWE.values()]

# semantic_search expects queries first, corpus second
hits = semantic_search(torch.tensor(CVE["description_embeddings"].to_list()), torch.tensor(category_embeddings), top_k=5)
# hits is a list of length len(CVE), each entry a list of top_k dicts:
# [{'corpus_id': int, 'score': float}, ...]

category_ids = list(CWE.keys()) # adjust key as needed

CVE["top_categories"] = [
    [category_ids[hit["corpus_id"]] for hit in row] for row in hits
]
CVE["top_category_scores"] = [
    [hit["score"] for hit in row] for row in hits
]

In [61]:
CVE.head(10)

,cve_id,published,last_modified,published_year,description,all_nvd_cwes,cwe_699_matches,number_of_nvd_cwes,number_of_cwe_699_matches,description_embeddings,top_categories,top_category_scores,matched_top_category
0,CVE-2025-0168,2025-01-01T14:15:23.590,2026-06-17T08:26:00.070,2025,A vulnerability classified as critical has bee...,CWE-74|CWE-89,CWE-89,2,1,"[-0.093904324, 0.006129023, -0.11886988, 0.024...","[1006, 389, 275, 417, 438]","[0.4787847399711609, 0.4699954390525818, 0.438...",False
1,CVE-2025-22214,2025-01-02T04:15:06.277,2026-06-17T08:45:38.530,2025,Landray EIS 2001 through 2006 allows Message/f...,CWE-89,CWE-89,1,1,"[0.029114302, -0.030592905, -0.060772974, 0.04...","[1214, 1228, 840, 1212, 19]","[0.22140471637248993, 0.2165326327085495, 0.19...",False
2,CVE-2025-0171,2025-01-02T15:15:25.550,2026-06-17T08:26:00.413,2025,"A vulnerability, which was classified as criti...",CWE-74|CWE-89,CWE-89,2,1,"[-0.06441131, 0.009685945, -0.0849804, -0.0064...","[275, 1217, 417, 389, 1006]","[0.4399569034576416, 0.42936667799949646, 0.42...",False
3,CVE-2025-0172,2025-01-02T16:15:09.103,2026-06-17T08:26:00.560,2025,A vulnerability has been found in code-project...,CWE-74|CWE-89,CWE-89,2,1,"[-0.048654113, 0.0076001417, -0.09157962, -0.0...","[275, 417, 389, 1217, 1214]","[0.44955188035964966, 0.44222205877304077, 0.4...",False
4,CVE-2025-0173,2025-01-02T18:15:21.630,2026-06-17T08:26:00.697,2025,A vulnerability was found in SourceCodester On...,CWE-74|CWE-89,CWE-89,2,1,"[-0.05857405, 0.034661274, -0.08664874, 0.0035...","[389, 1006, 438, 275, 569]","[0.43114209175109863, 0.42284291982650757, 0.4...",False
5,CVE-2025-0174,2025-01-03T01:15:08.100,2026-06-17T08:26:00.840,2025,A vulnerability was found in code-projects Poi...,CWE-74|CWE-89,CWE-89,2,1,"[-0.029808877, 0.056778476, -0.12597811, 0.038...","[389, 1006, 429, 438, 1214]","[0.5535783767700195, 0.4644908308982849, 0.456...",False
6,CVE-2025-0175,2025-01-03T01:15:08.263,2026-06-17T08:26:00.980,2025,A vulnerability was found in code-projects Onl...,CWE-79|CWE-94,CWE-79|CWE-94,2,2,"[-0.07794925, 0.04224955, -0.10277475, -0.0054...","[1006, 389, 438, 417, 275]","[0.4776696264743805, 0.4588359594345093, 0.449...",False
7,CVE-2025-0176,2025-01-03T02:15:07.870,2026-06-17T08:26:01.123,2025,A vulnerability was found in code-projects Poi...,CWE-74|CWE-89,CWE-89,2,1,"[-0.033388007, 0.06400511, -0.10010375, 0.0111...","[389, 275, 1006, 417, 320]","[0.5012896060943604, 0.4674651622772217, 0.464...",False
8,CVE-2025-21609,2025-01-03T17:15:09.147,2026-06-17T08:43:50.857,2025,"SiYuan is self-hosted, open source personal kn...",CWE-459|CWE-552,CWE-459|CWE-552,2,2,"[-0.05068957, 0.037208434, -0.027730467, -0.00...","[1214, 1219, 320, 417, 1210]","[0.26803430914878845, 0.2673332393169403, 0.23...",False
9,CVE-2025-21610,2025-01-03T17:15:09.290,2026-06-17T08:43:50.980,2025,Trix is a what-you-see-is-what-you-get rich te...,CWE-79,CWE-79,1,1,"[-0.10250606, -0.009428056, 0.010922128, 0.012...","[429, 417, 1215, 1228, 438]","[0.24495328962802887, 0.2309284806251526, 0.22...",False
